In [2]:
import pandas as pd
import numpy as np
def process_data(df):
    df = df.copy()

    # is_new
    df["is_new"] = (df["mileage"].astype(str).str.strip() == "Новый").astype(int)

    # mileage
    df["mileage"] = (
        df["mileage"]
        .astype(str)
        .str.strip()
        .str.replace("Новый", "0", regex=False)
        .str.replace(r"\D+", "", regex=True)
    )

    df["mileage"] = pd.to_numeric(df["mileage"], errors="coerce").fillna(0)

    # fuel
    df["fuel"] = (
        df["engine"]
        .astype(str)
        .str.extract(r"/\s*([^/]+)$")[0]
        .str.strip()
    )

    # engine volume
    df["engine_volume"] = (
        df["engine"]
        .astype(str)
        .str.extract(r"(\d+(?:\.\d+)?)\s*л")[0]
    )

    df["engine_volume"] = pd.to_numeric(df["engine_volume"], errors="coerce").fillna(0)

    # horse power
    df["horse_power"] = (
        df["engine"]
        .astype(str)
        .str.extract(r"(\d+)\s*л\.с")[0]
    )

    df["horse_power"] = pd.to_numeric(df["horse_power"], errors="coerce").fillna(0)

    df["car_age"] = 2023 - df["year"]

    df = df.drop(["engine"], axis=1)

    if "color" in df.columns:
        df = df.drop("color", axis=1)

    if "car_id" in df.columns:
        df = df.drop("car_id", axis=1)

    return df

In [3]:
import numpy as np
from catboost import CatBoostRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df_test = pd.read_csv('/Users/nurdauletaldibek/Desktop/FINAL_project/test.csv')
df_train = pd.read_csv('/Users/nurdauletaldibek/Desktop/FINAL_project/train.csv')

df_model = process_data(df_train)

y = df_model["price"]
X = df_model.drop("price", axis=1)

In [4]:
df = process_data(df_train)

print(df.isna().sum()[df.isna().sum() > 0])
print(np.isinf(df.select_dtypes(include=[np.number])).sum())
df

Series([], dtype: int64)
year             0
mileage          0
price            0
is_new           0
engine_volume    0
horse_power      0
car_age          0
dtype: int64


,car_mark,year,mileage,drive,body,transmission,brand,price,is_new,fuel,engine_volume,horse_power,car_age
0,Subaru Legacy I,1994,250000,полный,седан,автомат,Subaru,85000,0,Бензин,2.2,136,29
1,Daewoo Nexia I,1998,262000,передний,седан,механика,Daewoo,90000,0,Бензин,1.5,75,25
2,Mazda CX-5 I Рестайлинг,2016,95000,полный,внедорожник 5 дв.,автомат,Mazda,1695000,0,Бензин,2.0,150,7
3,Kia Seltos I,2022,0,передний,внедорожник 5 дв.,вариатор,Kia,2284900,1,Бензин,2.0,149,1
4,Renault Logan II Рестайлинг,2022,0,передний,седан,механика,Renault,921300,1,Бензин,1.6,82,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57061,Hyundai Solaris II,2019,47224,передний,седан,автомат,Hyundai,1079000,0,Бензин,1.6,123,4
57062,Mitsubishi Outlander III Рестайлинг 3,2022,0,полный,внедорожник 5 дв.,вариатор,Mitsubishi,2818000,1,Бензин,2.0,146,1
57063,Skoda Rapid II,2022,0,передний,лифтбек,механика,Skoda,1253050,1,Бензин,1.6,90,1
57064,Toyota Altezza,1999,170000,задний,седан,механика,Toyota,600000,0,Бензин,2.0,210,24


In [4]:
df_train.head()

,car_id,car_mark,year,mileage,engine,drive,body,color,transmission,brand,price
0,63407,Subaru Legacy I,1994,250 000 км,2.2 л / 136 л.с. / Бензин,полный,седан,серебристый,автомат,Subaru,85000
1,15174,Daewoo Nexia I,1998,262 000 км,1.5 л / 75 л.с. / Бензин,передний,седан,красный,механика,Daewoo,90000
2,40267,Mazda CX-5 I Рестайлинг,2016,95 000 км,2.0 л / 150 л.с. / Бензин,полный,внедорожник 5 дв.,белый,автомат,Mazda,1695000
3,31131,Kia Seltos I,2022,Новый,2.0 л / 149 л.с. / Бензин,передний,внедорожник 5 дв.,Prestige,вариатор,Kia,2284900
4,55806,Renault Logan II Рестайлинг,2022,Новый,1.6 л / 82 л.с. / Бензин,передний,седан,Life,механика,Renault,921300


In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import numpy as np

df_train_model = process_data(df_train)

y = np.log1p(df_train_model["price"])
X = df_train_model.drop("price", axis=1)

cat_cols = ["brand", "transmission", "drive", "car_mark", "body", "fuel"]

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("unknown")

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    use_best_model=True
)

pred_val_log = model.predict(X_val)

rmse_log = np.sqrt(np.mean((y_val - pred_val_log) ** 2))
print("Validation log RMSE:", rmse_log)

0:	learn: 1.0692990	test: 1.0647781	best: 1.0647781 (0)	total: 70ms	remaining: 2m 19s
200:	learn: 0.2443857	test: 0.2517986	best: 0.2517986 (200)	total: 2.16s	remaining: 19.4s
400:	learn: 0.2217666	test: 0.2343380	best: 0.2343380 (400)	total: 4.32s	remaining: 17.2s
600:	learn: 0.2092235	test: 0.2266231	best: 0.2266231 (600)	total: 6.38s	remaining: 14.9s
800:	learn: 0.2000194	test: 0.2218437	best: 0.2218437 (800)	total: 8.51s	remaining: 12.7s
1000:	learn: 0.1927394	test: 0.2186813	best: 0.2186813 (1000)	total: 10.8s	remaining: 10.8s
1200:	learn: 0.1865999	test: 0.2164052	best: 0.2164052 (1200)	total: 12.9s	remaining: 8.59s
1400:	learn: 0.1810674	test: 0.2143419	best: 0.2143419 (1400)	total: 15.1s	remaining: 6.44s
1600:	learn: 0.1763624	test: 0.2126012	best: 0.2126012 (1600)	total: 17.2s	remaining: 4.29s
1800:	learn: 0.1720997	test: 0.2111459	best: 0.2111459 (1800)	total: 19.3s	remaining: 2.14s
1999:	learn: 0.1681967	test: 0.2098358	best: 0.2098358 (1999)	total: 21.6s	remaining: 0us

bes

In [ ]:
X_test = process_data(df_test)

for col in cat_cols:
    X_test[col] = X_test[col].astype(str).fillna("unknown")

pred_log = model.predict(X_test)
predictions = np.expm1(pred_log)

predictions = np.clip(predictions, 10000, None)

submission = pd.DataFrame({
    "car_id": df_test["car_id"],
    "predicted_price": predictions
})

submission.to_csv("catboost_submission.csv", index=False)

In [142]:
X_test

,car_mark,year,mileage,drive,body,transmission,brand,is_new,fuel,engine_volume,horse_power,car_age
0,Volkswagen Tiguan II,2017,80277,полный,внедорожник 5 дв.,робот,Volkswagen,0,Дизель,2.0,150,6
1,Infiniti G G25 IV,2010,224000,задний,седан,автомат,Infiniti,0,Бензин,2.5,222,13
2,Hyundai Creta II,2022,0,передний,внедорожник 5 дв.,автомат,Hyundai,1,Бензин,2.0,150,1
3,Honda Civic VII,2001,194000,передний,седан,автомат,Honda,0,Бензин,1.7,125,22
4,Hyundai Creta I Рестайлинг,2021,15500,полный,внедорожник 5 дв.,механика,Hyundai,0,Бензин,1.6,121,2
...,...,...,...,...,...,...,...,...,...,...,...,...
19018,Volkswagen Polo V Рестайлинг,2015,139000,передний,седан,автомат,Volkswagen,0,Бензин,1.6,105,8
19019,Nissan Almera III (G15),2014,105000,передний,седан,механика,Nissan,0,Бензин,1.6,102,9
19020,Lexus LX 570 III Рестайлинг 2,2016,218000,полный,внедорожник 5 дв.,автомат,Lexus,0,Бензин,5.7,367,7
19021,Toyota Land Cruiser 300 Series,2022,16,полный,внедорожник 5 дв.,автомат,Toyota,0,Дизель,3.4,299,1
